Lets focus on basic data analysis and the cities locations. 

In [1]:
# Libraries import

import requests
import json
import time
from dotenv import load_dotenv
import os
import pandas as pd

# Configuration

pd.set_option('display.max_columns', None) # display each column's dataframe

# 1. Call api

## Call api nominatim

In [2]:
# import requests

nominatim_url = "https://nominatim.openstreetmap.org/search"
cities_info = f"{nominatim_url}/cities_location.json"

params = {
    "country": "France",
    "city": "Bayeux",
    "format": "geocodejson"
}

headers = {"User-Agent": "projet_etude_dataviz (lnmourez@gmail.com)"}

### Fetch cities information

In [3]:
# import json

nominatim_response = requests.get(nominatim_url, params=params, headers=headers) # call api

print(nominatim_response) # call api status code 
print(nominatim_response.content) # take a look at the data that was recieved along with this response
#print(json.dumps(response.json(), indent=2)) # add json indentation



<Response [200]>
b'{"type":"FeatureCollection","geocoding":{"version":"0.1.0","attribution":"Data \xc2\xa9 OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright","licence":"ODbL","query":"Bayeux, France"},"features":[{"type":"Feature","properties":{"geocoding":{"place_id":284771403,"osm_type":"relation","osm_id":145776,"osm_key":"boundary","osm_value":"administrative","type":"city","label":"Bayeux, Calvados, Normandie, France m\xc3\xa9tropolitaine, 14400, France","name":"Bayeux"}},"geometry":{"type": "Point","coordinates": [-0.7024738, 49.2764624]}},{"type":"Feature","properties":{"geocoding":{"place_id":284516418,"osm_type":"relation","osm_id":1656892,"osm_key":"boundary","osm_value":"administrative","type":"city","label":"Bayeux, Calvados, Normandie, France m\xc3\xa9tropolitaine, France","name":"Bayeux"}},"geometry":{"type": "Point","coordinates": [-0.791677, 49.2455634]}}]}'


In [4]:
type(nominatim_response.json())  # display type

dict

#### Fetch one city location

In [5]:
feature = nominatim_response.json()["features"][0] # declare the first result
city = feature["properties"]["geocoding"]["name"]
place_id = feature["properties"]["geocoding"]["place_id"]
lat = feature["geometry"]["coordinates"][1]
lon = feature["geometry"]["coordinates"][0]
response = requests.get(nominatim_url, params=params, headers=headers)
print()
print(f"{city}, {place_id},{lon}, {lat}")


Bayeux, 284771403,-0.7024738, 49.2764624


#### Fetch the cities list location

In the provided cities list, we observe that :
1) Ariège is a department. 
2) Gorges du Verdon and Chateau du haut Koenigsbourg are tourist sites

It could be interesting to search some nearby locations. 

Ariège :
- https://www.guide-toulouse-pyrenees.com/fr/experiences/culturelle/article-les-plus-beaux-villages-de-l-ariege-106.html
- https://www.sites-touristiques-ariege.fr/informations-pratiques/visiter-lariege/

Bédeilhac-et-Aynat
Camon
Carla-Bayle
Foix
Le Mas-d’Azil
Mirepoix
Montségur
Niaux
Saint-Lizier
Saint-Martin-d’Oydes
Tarascon-sur-Ariège

Gorges du Verdon : 
https://lesgorgesduverdon.fr/a-visiter/

Barjols
Castellane
Cotignac
Moustiers-Sainte-Marie
Sainte-Croix-du-Verdon
Sillans-la-Cascade
Valensole

Chateau du haut Koenigsbourg (ORSCHWILLER)
https://www.haut-koenigsbourg.fr/infos-pratiques/

In [ ]:
# import time

cities_info = []
cities = ["Aigues Mortes",
"Aix en Provence",
"Amiens",
"Annecy",
"Avignon",
"Barjols",
"Bayeux",
"Bayonne",
"Besancon",
"Biarritz",
"Bormes les Mimosas",
"Bédeilhac-et-Aynat",
"Camon",
"Carcassonne",
"Carla-Bayle",
"Cassis",
"Castellane",
"Collioure",
"Colmar",
"Cotignac",
"Dijon",
"Eguisheim",
"Foix",
"Grenoble",
"La Rochelle",
"Le Havre",
"Le Mas-d’Azil",
"Lille",
"Lyon",
"Marseille",
"Mirepoix",
"Mont Saint Michel",
"Montauban",
"Montségur",
"Moustiers-Sainte-Marie",
"Niaux",
"Nimes",
"Orschwiller",
"Paris",
"Rouen",
"Saint-Lizier",
"Saint-Martin-d’Oydes",
"Sainte-Croix-du-Verdon",
"Saintes Maries de la mer",
"Sillans-la-Cascade",
"St Malo",
"Strasbourg",
"Tarascon-sur-Ariège",
"Toulouse",
"Uzes",
"Valensole"
]

for city in cities:
    try:
        nominatim_response = requests.get(
            nominatim_url, 
            params={"country": "France",
            "city": city,
            "format": "geocodejson"
            },
            headers=headers)
        # print(f"{city}: {response.status_code}")  # vérifiez le code HTTP


        feature = nominatim_response.json()["features"][0]
        cities_info.append({"city" : feature["properties"]["geocoding"]["name"],
        "place_id" : feature["properties"]["geocoding"]["place_id"],
        "lat" : feature["geometry"]["coordinates"][1],
        "lon" : feature["geometry"]["coordinates"][0]
        })
    except (IndexError, KeyError, requests.exceptions.RequestException):
        print(f"Execution with errors for {city} : {nominatim_response.status_code}")
      
    time.sleep(2) 


In [7]:
# import json
print(len(cities_info))
print(cities_info)
#print(json.dumps(cities_location, indent=2))

2
[{'city': 'Le Carla-Bayle', 'place_id': 85080454, 'lat': 43.1500386, 'lon': 1.3936011}, {'city': 'Valensole', 'place_id': 82788602, 'lat': 43.8379283, 'lon': 5.9839867}]


In [8]:
type(cities_info)

list

## 2. Call api openweathermap

In [9]:
# from dotenv import load_dotenv
# import os
# V2.5 https://api.openweathermap.org/data/2.5/weather
# V4.0 https://api.openweathermap.org/data/4.0/onecall/current

weather_url = "https://api.openweathermap.org/data/2.5/weather"
params = {
    "lat":"48.635954",
    "lon":"-1.511460",
    "units":"metric",
    "appid": os.getenv("API_KEY")
}

### Fetch weather information

In [10]:
weather_response = requests.get(weather_url, params=params) # call api
weather_json = weather_response.json()

print(weather_response, weather_json)

<Response [200]> {'coord': {'lon': -1.5115, 'lat': 48.636}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 20.48, 'feels_like': 20.65, 'temp_min': 20.48, 'temp_max': 20.48, 'pressure': 1009, 'humidity': 79, 'sea_level': 1009, 'grnd_level': 1004}, 'visibility': 10000, 'wind': {'speed': 1.4, 'deg': 328, 'gust': 1.37}, 'clouds': {'all': 100}, 'dt': 1787764290, 'sys': {'country': 'FR', 'sunrise': 1787721275, 'sunset': 1787770901}, 'timezone': 7200, 'id': 6435453, 'name': 'Huisnes-sur-Mer', 'cod': 200}


In [11]:
weather_info = []
for city in cities_info:
    try:
        params = {
            "lat": city["lat"], 
            "lon": city["lon"], 
            "units": "metric", 
            "appid": "65a6acbff3eb4de13c8e4cf52a30ccf2"
            }
        weather_response = requests.get(weather_url, params=params)
        weather = weather_response.json()
        weather['city'] = city['city'] #weather_info.append({"city": city["city"], "weather": weather_response.json()})
        weather_info.append(weather)
    except (IndexError, KeyError, requests.exceptions.RequestException):
        print(f"Execution with errors for {city} : {weather_response.status_code}")
    time.sleep(2)

In [12]:
print(cities_info[0].keys())  # display keys

dict_keys(['city', 'place_id', 'lat', 'lon'])


In [13]:
print(weather_info[0].keys())  # display keys

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod', 'city'])


In [14]:
# print(json.dumps(weather_info, indent=2))

In [15]:
type(weather_info)

list

In [16]:
weather_info

[{'coord': {'lon': 1.3936, 'lat': 43.15},
  'weather': [{'id': 801,
    'main': 'Clouds',
    'description': 'few clouds',
    'icon': '02d'}],
  'base': 'stations',
  'main': {'temp': 33.97,
   'feels_like': 31.43,
   'temp_min': 31.22,
   'temp_max': 34.66,
   'pressure': 1006,
   'humidity': 10,
   'sea_level': 1006,
   'grnd_level': 967},
  'visibility': 10000,
  'wind': {'speed': 2.66, 'deg': 220, 'gust': 4.44},
  'clouds': {'all': 20},
  'dt': 1787764607,
  'sys': {'type': 2,
   'id': 48226,
   'country': 'FR',
   'sunrise': 1787721121,
   'sunset': 1787769661},
  'timezone': 7200,
  'id': 6446851,
  'name': 'Artigat',
  'cod': 200,
  'city': 'Le Carla-Bayle'},
 {'coord': {'lon': 5.984, 'lat': 43.8379},
  'weather': [{'id': 804,
    'main': 'Clouds',
    'description': 'overcast clouds',
    'icon': '04d'}],
  'base': 'stations',
  'main': {'temp': 25.76,
   'feels_like': 26.09,
   'temp_min': 25.76,
   'temp_max': 25.76,
   'pressure': 1014,
   'humidity': 65,
   'sea_level': 10

# 2. Convert list to DataFrame

## cities dataframe convertion

In [17]:
# import pandas as pd

cities_df = pd.DataFrame(cities_info)

## weather normalization

In [18]:
weather_df = pd.json_normalize(weather_info)


In [19]:
type(cities_df)

pandas.core.frame.DataFrame

In [20]:
type(weather_df)

pandas.core.frame.DataFrame

In [21]:
cities_df.head(1)

,city,place_id,lat,lon
0,Le Carla-Bayle,85080454,43.150039,1.393601


In [22]:
type(cities_df)

pandas.core.frame.DataFrame

In [23]:
type(cities_df['city'])

pandas.core.series.Series

In [24]:
cities_df['city'].tolist()

['Le Carla-Bayle', 'Valensole']

In [25]:
type(cities_df['city'].tolist())

list

In [26]:
print(cities_df['city'].tolist())

['Le Carla-Bayle', 'Valensole']


In [27]:
weather_df.head(1)

,weather,base,visibility,dt,timezone,id,name,cod,city,coord.lon,coord.lat,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.humidity,main.sea_level,main.grnd_level,wind.speed,wind.deg,wind.gust,clouds.all,sys.type,sys.id,sys.country,sys.sunrise,sys.sunset
0,"[{'id': 801, 'main': 'Clouds', 'description': ...",stations,10000,1787764607,7200,6446851,Artigat,200,Le Carla-Bayle,1.3936,43.15,33.97,31.43,31.22,34.66,1006,10,1006,967,2.66,220,4.44,20,2.0,48226.0,FR,1787721121,1787769661


In [28]:
weather = pd.json_normalize(weather_df['weather'])

In [29]:
weather.head(1)

,0
0,"{'id': 801, 'main': 'Clouds', 'description': '..."


In [30]:
weather2 = pd.json_normalize(weather.to_dict(orient="records"))
weather2 = weather2.add_prefix("data.")

In [31]:
weather2.head(1)

,data.0.id,data.0.main,data.0.description,data.0.icon
0,801,Clouds,few clouds,02d


In [32]:
weather_df = pd.concat(
    [weather_df.drop(columns="weather"),
     weather2],
     axis=1)

In [33]:
# pd.set_option('display.max_columns', None) # display each column's dataframe
weather_df.head(1)

,base,visibility,dt,timezone,id,name,cod,city,coord.lon,coord.lat,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.humidity,main.sea_level,main.grnd_level,wind.speed,wind.deg,wind.gust,clouds.all,sys.type,sys.id,sys.country,sys.sunrise,sys.sunset,data.0.id,data.0.main,data.0.description,data.0.icon
0,stations,10000,1787764607,7200,6446851,Artigat,200,Le Carla-Bayle,1.3936,43.15,33.97,31.43,31.22,34.66,1006,10,1006,967,2.66,220,4.44,20,2.0,48226.0,FR,1787721121,1787769661,801,Clouds,few clouds,02d


In [34]:
weather_df.columns

Index(['base', 'visibility', 'dt', 'timezone', 'id', 'name', 'cod', 'city',
       'coord.lon', 'coord.lat', 'main.temp', 'main.feels_like',
       'main.temp_min', 'main.temp_max', 'main.pressure', 'main.humidity',
       'main.sea_level', 'main.grnd_level', 'wind.speed', 'wind.deg',
       'wind.gust', 'clouds.all', 'sys.type', 'sys.id', 'sys.country',
       'sys.sunrise', 'sys.sunset', 'data.0.id', 'data.0.main',
       'data.0.description', 'data.0.icon'],
      dtype='object')

# 3. Merge DataFrames on 'city'

In [35]:
current_weather = pd.merge(cities_df, weather_df, on='city')

# Display key columns
print("\nCurrent weather:")
current_weather.head()


Current weather:


,city,place_id,lat,lon,base,visibility,dt,timezone,id,name,cod,coord.lon,coord.lat,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.humidity,main.sea_level,main.grnd_level,wind.speed,wind.deg,wind.gust,clouds.all,sys.type,sys.id,sys.country,sys.sunrise,sys.sunset,data.0.id,data.0.main,data.0.description,data.0.icon
0,Le Carla-Bayle,85080454,43.150039,1.393601,stations,10000,1787764607,7200,6446851,Artigat,200,1.3936,43.1500,33.97,31.43,31.22,34.66,1006,10,1006,967,2.66,220,4.44,20,2.0,48226.0,FR,1787721121,1787769661,801,Clouds,few clouds,02d
1,Valensole,82788602,43.837928,5.983987,stations,10000,1787764294,7200,2971033,Valensole,200,5.9840,43.8379,25.76,26.09,25.76,25.76,1014,65,1014,957,2.84,151,5.80,91,NaN,NaN,FR,1787719956,1787768623,804,Clouds,overcast clouds,04d


In [36]:
type(current_weather)

pandas.core.frame.DataFrame

In [42]:
current_weather.loc[current_weather["city"] == "Carla-Bayle"].iloc[0, 0]
# current_weather["city"] = current_weather["city"].replace("Le Carla-Bayle", "Carla-Bayle")

'Carla-Bayle'

In [44]:
print(current_weather['city'])

0    Carla-Bayle
1      Valensole
Name: city, dtype: object
